# 14 · Capstone: demonstrate an order pipeline and its recovery

**Level:** advanced/expert practice · **Time:** 60–90 minutes · **Needs:** notebooks 11–13 and Kafka.

Demonstrate the complete path: HTTP request → order + outbox → Kafka → consumer receipt +
projection → HTTP read. Prove eventual consistency, simulate a publisher crash, run workers
as separate processes, and verify business invariants. Use the checklist at the end as
a presentation script. Passing this lab is a starting point for production design, not
a claim that the local single-broker system is production-ready.

Restart the kernel and Run All. Topics, group IDs, orders, and database are isolated per run.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


### Jupyter and synchronous Django code

Jupyter kernels may keep an event loop running even with `%autoawait off`. The next cell
installs `%%django_sync`: it executes the visible Python body on one dedicated synchronous
thread and waits for completion. All Django code cells below use that thread, so ORM
connection ownership and transaction boundaries remain consistent. Variables stay in the
notebook namespace, exceptions remain visible, and Django's async safety stays enabled.
The magic adds no business logic. In a terminal, run the same Python bodies normally.


In [ ]:
%autoawait off
sys.path.insert(0, str(ROOT / "django_demo"))
from notebook_support import install_sync_magic
DJANGO_EXECUTOR = install_sync_magic(get_ipython())
print("Synchronous Django cell executor installed")


In [ ]:
%%django_sync
LESSON = "14"

import os
import django
from django.conf import settings

if settings.configured:
    raise RuntimeError("Restart the kernel before running this notebook again; Django settings are process-wide.")
DATA_DIR = ROOT / ".lab-data"
DATA_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / f"django-{LESSON}-{RUN}.sqlite3"
os.environ["DJANGO_DB_PATH"] = str(DB_PATH)
os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
sys.path.insert(0, str(ROOT / "django_demo"))
django.setup()
from django.core.management import call_command
from django.db import transaction
from django.test import Client, override_settings
from django.utils import timezone
from decimal import Decimal
from unittest.mock import patch
from orders.models import Order, OutboxEvent, InboxEvent, OrderProjection
from orders.services import (
    order_event, validate_event, publish_event, apply_event,
    create_order_after_commit, create_order_with_outbox, publish_pending,
)
call_command("migrate", verbosity=0)
http = Client()
print("Isolated database:", DB_PATH.name)


## Step 1 — Set up a three-partition event stream

Keys are order IDs, so all records for one order use one partition with a stable partition
count. Different orders can be consumed in parallel; no global order is promised.
This capstone still runs one consumer process to keep the demonstration easy to observe.


In [ ]:
%%django_sync
require_broker()
topic = new_topic("capstone-orders", partitions=3)
group = f"lab.capstone.{RUN}"
print("Topic:", topic, "Consumer group:", group)


## Step 2 — Receive five HTTP requests

The database commits order state and outgoing intent together. Kafka is not needed
inside these requests. Keep the returned IDs: they connect the HTTP, outbox, Kafka,
receipt, and projection views of the same work.


In [ ]:
%%django_sync
accepted = []
with override_settings(KAFKA_ORDERS_TOPIC=topic):
    for customer, total in [("Ada", "10.00"), ("Grace", "20.25"), ("Linus", "5.50"), ("Margaret", "30.00"), ("Guido", "14.25")]:
        response = http.post("/orders/", {"customer": customer, "total": total}, content_type="application/json")
        assert response.status_code == 201, response.content
        accepted.append(response.json())
assert Order.objects.count() == OutboxEvent.objects.count() == 5
assert OutboxEvent.objects.filter(published_at__isnull=True).count() == 5
print(accepted)


## Step 3 — Observe eventual consistency

The write model exists while all read models are absent. This is expected pipeline
delay. A UI can show 'processing' until a projection appears; it must not equate request
acceptance with completion of every downstream side effect.


In [ ]:
%%django_sync
initial_statuses = [http.get(f'/projections/{item["order_id"]}/').status_code for item in accepted]
assert initial_statuses == [404] * 5
print("Before workers:", initial_statuses)


## Step 4 — Simulate an interrupted publisher

Deliver one pending event successfully, then leave its row pending. This is the same
externally visible state as a process crash between the Kafka acknowledgement and
the database marker update. It is intentional and will create one duplicate.


In [ ]:
%%django_sync
interrupted = OutboxEvent.objects.order_by("created_at", "event_id").first()
publish_event(interrupted.payload, interrupted.topic)
assert interrupted.published_at is None
print("Delivered but unmarked:", interrupted.event_id)


## Step 5 — Run a replacement outbox publisher process

`subprocess.run()` inherits `DJANGO_DB_PATH` set above. The process uses the same local
database, explicit topic filter, and Python environment as the notebook. Each worker
is bounded, and failures propagate through `check=True`.


In [ ]:
%%django_sync
import subprocess

def worker(command, *arguments):
    completed = subprocess.run(
        [sys.executable, str(ROOT / "django_demo" / "manage.py"), command, *arguments],
        cwd=ROOT, env=os.environ.copy(), text=True, capture_output=True,
        timeout=120, check=True,
    )
    print(completed.stdout.strip())
    return completed

worker("publish_outbox", "--topic", topic, "--max-messages", "10", "--timeout", "60")
assert OutboxEvent.objects.filter(published_at__isnull=True).count() == 0


## Step 6 — Inspect the event log before applying it

A fresh inspection group reads six Kafka records: five events, one delivered twice.
Independent groups do not steal messages from each other. Partition/offset identifies
a log record; event ID identifies the business fact that must be applied once.


In [ ]:
%%django_sync
messages = collect(topic, count=6)
inspected_ids = [decode(message)["event_id"] for message in messages]
assert len(set(inspected_ids)) == 5
for message in messages:
    value = decode(message)
    print(message.partition(), message.offset(), value["event_id"], value["data"]["customer"])


## Step 7 — Run the projection consumer process

Each message is validated, then its receipt and projection are committed together.
Only after that database transaction succeeds does the consumer acknowledge the Kafka
position. The repeated event is a successful no-op and can safely be acknowledged.


In [ ]:
%%django_sync
worker("consume_orders", "--topic", topic, "--group", group, "--max-messages", "6", "--timeout", "40")
assert InboxEvent.objects.count() == 5
assert OrderProjection.objects.count() == 5


## Step 8 — Verify business results through HTTP

Validate more than process exit codes: every accepted order should have one projection,
the event IDs should match, and exact-decimal totals should agree.


In [ ]:
%%django_sync
projected = []
for item in accepted:
    response = http.get(f'/projections/{item["order_id"]}/')
    assert response.status_code == 200
    body = response.json()
    assert body["source_event_id"] == item["event_id"]
    assert Decimal(body["total"]) == Decimal(item["total"])
    projected.append(body)
expected_total = sum((Decimal(item["total"]) for item in accepted), Decimal("0.00"))
actual_total = sum((row.total for row in OrderProjection.objects.all()), Decimal("0.00"))
assert expected_total == actual_total == Decimal("80.00")
print("Five projections; total:", actual_total)


## Step 9 — Rebuild or replay with a new group

A new group starts from the beginning, but the existing receipt table still remembers
each event. All six deliveries should be duplicates. To rebuild a projection into a
genuinely new store, use a separate database (including fresh receipts); do not clear
receipts while a live consumer is using them.


In [ ]:
%%django_sync
worker("consume_orders", "--topic", topic, "--group", f"lab.capstone-replay.{RUN}", "--max-messages", "6", "--timeout", "40")
assert InboxEvent.objects.count() == OrderProjection.objects.count() == 5
print("Replay preserved every business invariant")


## Step 10 — Calculate operational indicators

Inspect backlog and oldest pending age in the database. Kafka group lag is a different
measurement: notebook 10 covers log offsets and consumer positions. Low outbox backlog
does not prove that downstream consumers are healthy.


In [ ]:
%%django_sync
pending_rows = OutboxEvent.objects.filter(published_at__isnull=True)
oldest = pending_rows.order_by("created_at").first()
oldest_age_seconds = (timezone.now() - oldest.created_at).total_seconds() if oldest else 0
print({"outbox_pending": pending_rows.count(), "oldest_pending_seconds": oldest_age_seconds,
       "orders": Order.objects.count(), "projected_orders": OrderProjection.objects.count()})
assert pending_rows.count() == 0


## Your live demonstration script

1. Explain topics, partitions, keys, offsets, and group IDs using the six inspected records.
2. Pause before starting workers and show the five 404 projection responses.
3. Explain the deliberately unmarked event and predict the number of Kafka records.
4. Start each worker and show its bounded output.
5. Show six deliveries becoming five database effects, then replay without changing totals.
6. Name the remaining boundaries: HTTP request retries, broker replication, authentication,
   schemas, retention, poison records, and monitoring.

## Expert extension challenges

- Add `order.cancelled` with explicit allowed transitions and per-order sequence numbers.
- Partition work across two consumer processes and verify exclusive assignments.
- Move to PostgreSQL and design publisher ownership, retries, and ordering under concurrency.
- Add a DLQ and a reviewed redrive operation that preserves event IDs.
- Measure p50/p95 end-to-end delay and alarm on oldest pending outbox age.

**Solution direction:** write the invariants and crash matrix first. A retry must keep
event identity; a transition must reject stale versions; concurrent publisher ownership
must expire safely. Use notebook 15 to explore broker replication separately. Keep this
SQLite, single-publisher, unauthenticated local application scoped to teaching.
